In [1]:
import sys
sys.path.append('../../../')

In [2]:
import os
os.chdir('../../../')

In [7]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import QED
from rdkit.Chem.Draw import MolsToGridImage

from cgflow.util.rdkit import calculate_sa_score
from app.api.sbdd_sample import sample_against_pockets
from app.api.sbdd_sample import build_pocket_conditional_sampler

NOTE: Please refer to README.md for details on how to download the pretrained model weights.

In [ ]:
POSE_CKPT_PATH = "crossdocked2020_till_end.ckpt"
CGFLOW_CKPT_PATH = "sbdd_proxy_iter_32000_emb.pt"
POCKET_PATHS = [
    "crossdocked_pocket10/GNA13_MOUSE_28_377_GTP_0/1zcb_A_rec_1zcb_gdp_lig_tt_docked_0_pocket10.pdb"
]

In [4]:
import torch
pose_state = torch.load(POSE_CKPT_PATH, map_location='cuda', weights_only=False)
cgflow_state = torch.load(CGFLOW_CKPT_PATH, map_location='cuda', weights_only=False)


In [5]:
pose_state['cfg'] = cgflow_state['cfg']
pose_state['models_state_dict'] = cgflow_state['models_state_dict']
torch.save(pose_state, POSE_CKPT_PATH)

In [ ]:
sampler = build_pocket_conditional_sampler(
    ckpt_path=POSE_CKPT_PATH,
    temperature=(48, 64),
    num_samples=100,
    device="cuda",
    seed=1,
)

In [8]:
output = sample_against_pockets(
    sampler=sampler,
    pocket_paths=POCKET_PATHS,
    num_samples=100,
    seed=1,
)

In [9]:
results = output[0][0]

In [11]:
mols = [Chem.MolFromSmiles(Chem.MolToSmiles(mol['mol'])) for mol in results]
sas = [calculate_sa_score(mol) for mol in mols]
qeds = [QED.qed(mol) for mol in mols]